Проверка, что пайплайн рабочий

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("sapientinc/HRM-Text-1B")
model = AutoModelForCausalLM.from_pretrained(
    "sapientinc/HRM-Text-1B",
    dtype=torch.float32,
    attn_implementation="sdpa",
).to("cuda").eval()

inputs = tokenizer("The quick brown fox", return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=16, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

The quick brown fox fox fox fox fox fox fox fox fox fox fox fox fox fox fox fox fox


Модель просто зацикливается на слове "Fox". Пробуем промпт с PrefixLM-разметкой

In [3]:
condition = "<|quad_end|><|object_ref_end|>"
prompt = f"<|im_start|>{condition}Explain why the sky is blue.<|im_end|>"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
inputs["token_type_ids"] = torch.ones_like(inputs["input_ids"])

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=False))

<|im_start|><|quad_end|><|object_ref_end|>Explain why the sky is blue.<|im_end|>The sky appears blue due to the scattering of sunlight by air molecules. Shorter wavelengths of light (blue) scatter more easily than longer wavelengths (red), which is why the sky appears blue. This phenomenon is known as Rayleigh scattering. $\boxed{\text{The sky appears blue because shorter wavelengths of light (blue) scatter more easily than longer wavelengths (red) due to air molecules.}}$<|box_end|>


Получили уже более связанный ответ, отлично. Раз базовый pipeline работает, переходим к первому эксперименту — проверим как модель ведёт себя с retrieved context

**Experiment 1**

In [4]:
context = "The Eiffel Tower was completed in 1889 and is located in Paris, France."
question = "In what year was the Eiffel Tower completed?"

condition = "<|quad_end|><|object_ref_end|>"
prompt = f"<|im_start|>{condition}Context: {context}\n\nQuestion: {question}\nAnswer:<|im_end|>"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
inputs["token_type_ids"] = torch.ones_like(inputs["input_ids"])

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=50, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=False))

<|im_start|><|quad_end|><|object_ref_end|>Context: The Eiffel Tower was completed in 1889 and is located in Paris, France.

Question: In what year was the Eiffel Tower completed?
Answer:<|im_end|>1889<|box_end|>


Модель дала правильный ответ без лишних слов! PrefixLM отработал корректно! Теперь попробуем задать тот же самый вопрос, но без retrieval

**Experiment 2**

In [5]:
condition = "<|quad_end|><|object_ref_end|>"
prompt = f"<|im_start|>{condition}Question: In what year was the Eiffel Tower completed?\nAnswer:<|im_end|>"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
inputs["token_type_ids"] = torch.ones_like(inputs["input_ids"])

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=30, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=False))

<|im_start|><|quad_end|><|object_ref_end|>Question: In what year was the Eiffel Tower completed?
Answer:<|im_end|>The Eiffel Tower was completed in 1889. It was designed by French engineer Gustave Eiffel and constructed by the French company Eiffel


Без retrieval модель начала генерировать развёрнутый рассказ, добавила детали которых не было в вопросе. При этом ответ на вопрос - абсолютно корректный. Но год постройки Эйфелевой башни крайне известный факт. Попробуем взять несуществующий факт и сравнить результаты вопросов с контекстом и без.

**Experiment 3**

In [6]:
condition = "<|quad_end|><|object_ref_end|>"

context = "The Zylo Tower was completed in 2003 and is located in Meridia."
question = "In what year was the Zylo Tower completed?"

# Тест А — без контекста
prompt_no_context = f"<|im_start|>{condition}Question: {question}\nAnswer:<|im_end|>"

# Тест Б — с контекстом
prompt_with_context = f"<|im_start|>{condition}Context: {context}\n\nQuestion: {question}\nAnswer:<|im_end|>"

for name, prompt in [("БЕЗ контекста", prompt_no_context), ("С контекстом", prompt_with_context)]:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    inputs["token_type_ids"] = torch.ones_like(inputs["input_ids"])
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
    print(f"--- {name} ---")
    print(tokenizer.decode(out[0], skip_special_tokens=False))
    print()

--- БЕЗ контекста ---
<|im_start|><|quad_end|><|object_ref_end|>Question: In what year was the Zylo Tower completed?
Answer:<|im_end|>The Zylo Tower was completed in 1978. It stands at 1,042 meters (3,420 feet) and is the tallest building

--- С контекстом ---
<|im_start|><|quad_end|><|object_ref_end|>Context: The Zylo Tower was completed in 2003 and is located in Meridia.

Question: In what year was the Zylo Tower completed?
Answer:<|im_end|>2003<|box_end|>



Это отличный, показательный результат.

Без контекста — модель уверенно галлюцинирует: ни один из этих фактов не существует — Zylo Tower вымышленная башня. Важно — модель не сказала "я не знаю", она сгенерировала правдоподобно звучащий, детальный, но полностью выдуманный ответ.

С контекстом — идеально точный ответ, 2003.

Попробуем еще проверить результат на нескольких cгенерированных парах.

In [7]:
fake_facts = [
    ("The Vintra Bridge was built in 1956 and spans the Kelso River.", "In what year was the Vintra Bridge built?"),
    ("Dr. Elena Foss discovered the mineral Rathenite in 1887.", "Who discovered Rathenite?"),
    ("The city of Norvenna has a population of 340,000 people.", "What is the population of Norvenna?"),
]

In [8]:
for context, question in fake_facts:
  # Тест А — без контекста
  prompt_no_context = f"<|im_start|>{condition}Question: {question}\nAnswer:<|im_end|>"

  # Тест Б — с контекстом
  prompt_with_context = f"<|im_start|>{condition}Context: {context}\n\nQuestion: {question}\nAnswer:<|im_end|>"

  for name, prompt in [("БЕЗ контекста", prompt_no_context), ("С контекстом", prompt_with_context)]:
      inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
      inputs["token_type_ids"] = torch.ones_like(inputs["input_ids"])
      with torch.no_grad():
          out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
      print(f"--- {name} ---")
      print(tokenizer.decode(out[0], skip_special_tokens=False))
      print()

--- БЕЗ контекста ---
<|im_start|><|quad_end|><|object_ref_end|>Question: In what year was the Vintra Bridge built?
Answer:<|im_end|>The Vintra Bridge was built in 1958. It is a 1,100-meter-long suspension bridge that connects the city of Vintra to t

--- С контекстом ---
<|im_start|><|quad_end|><|object_ref_end|>Context: The Vintra Bridge was built in 1956 and spans the Kelso River.

Question: In what year was the Vintra Bridge built?
Answer:<|im_end|>1956<|box_end|>

--- БЕЗ контекста ---
<|im_start|><|quad_end|><|object_ref_end|>Question: Who discovered Rathenite?
Answer:<|im_end|>To determine who discovered Rathenite, let's break it down step by step:

1. **Understand Rathenite**: Rathenite is a rare mineral with the chemical

--- С контекстом ---
<|im_start|><|quad_end|><|object_ref_end|>Context: Dr. Elena Foss discovered the mineral Rathenite in 1887.

Question: Who discovered Rathenite?
Answer:<|im_end|>Dr. Elena Foss<|box_end|>

--- БЕЗ контекста ---
<|im_start|><|quad_end|><|o

100% точность с контекстом на всех трёх примерах